# Qwen-Image-2.1 + Heretic Text Encoder (Gradio)

One-click Colab demo: **Qwen/Qwen-Image-2.1** with the community **Heretic (abliterated)** text encoder, Gradio UI, and a temporary public share link.

## Important notes

| Topic | Detail |
|-------|--------|
| **GGUF vs this notebook** | The GGUF on [`pottokao/Qwen-Image-2.1-Text-Encoder-Heretic-GGUF`](https://huggingface.co/pottokao/Qwen-Image-2.1-Text-Encoder-Heretic-GGUF) is **ComfyUI-only**. This notebook uses the **transformers bf16 shards**: [`pottokao/Qwen-Image-2.1-Text-Encoder-Heretic`](https://huggingface.co/pottokao/Qwen-Image-2.1-Text-Encoder-Heretic). |
| **GPU required** | Runtime → Change runtime type → **T4 / L4 / A100** (GPU). Free T4 (~15 GB) needs `enable_model_cpu_offload()`. |
| **First-run download** | Expect **~25–35 GB+** (base DiT/VAE + Heretic TE). Plan for 15–40+ minutes on Colab free. |
| **Default resolution** | Official default is **2048×2048** (VRAM-heavy). This UI defaults to **1024×1024** for free Colab. |
| **Share link** | `demo.launch(share=True)` prints a temporary `*.gradio.live` URL (~72 hours). Not permanent. |
| **Official free demos** | Blog: [qwen.ai/blog?id=qwen-image-2.1](https://qwen.ai/blog?id=qwen-image-2.1). Mainland CN free UI: [wuli.art](https://wuli.art/explore). |

**Run:** Runtime → **Run all** (top to bottom). After the last cell, copy the Gradio public URL from the output.


## 1. Runtime GPU check


In [ ]:
# Must show a CUDA device. If not: Runtime → Change runtime type → GPU → Save, then re-run.
import subprocess, sys

print("Python:", sys.version)
try:
    import torch
    print("torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        props = torch.cuda.get_device_properties(0)
        print(f"VRAM: {props.total_memory / 1024**3:.1f} GB")
    else:
        raise SystemExit("No GPU. Enable a GPU runtime before continuing.")
except ImportError:
    print("torch not installed yet — will be installed in the next cell.")
    # Still check nvidia-smi for a sanity check
    r = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
    print(r.stdout or r.stderr)
    if r.returncode != 0:
        raise SystemExit("nvidia-smi failed — enable a GPU runtime.")


## 2. Install dependencies (one cell)


In [ ]:
# Official stack + Gradio. Restart runtime only if Colab asks you to.
# Pin torchao so diffusers main can import FqnToConfig (avoids ImportError on older torchao).
%pip install -q "torch>=2.4.0" "transformers>=5.17" accelerate pillow gradio spaces
%pip install -q -U "torchao>=0.15.0"
%pip install -q git+https://github.com/huggingface/diffusers

import torch, transformers, diffusers, gradio, torchao
print("torch", torch.__version__)
print("transformers", transformers.__version__)
print("diffusers", diffusers.__version__)
print("torchao", getattr(torchao, "__version__", "?"))
print("gradio", gradio.__version__)
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A")
# Sanity: FqnToConfig must import (diffusers pulls it at import time on some builds)
from torchao.quantization import FqnToConfig
print("FqnToConfig OK", FqnToConfig)


## 3. Load Qwen-Image-2.1 + Heretic text encoder

Uses `bfloat16` + `enable_model_cpu_offload()` so free Colab T4 can run without OOM.


In [ ]:
import torch
from diffusers import QwenImage21Pipeline

BASE_MODEL = "Qwen/Qwen-Image-2.1"
HERETIC_TE = "pottokao/Qwen-Image-2.1-Text-Encoder-Heretic"
DTYPE = torch.bfloat16

# Import Qwen3-VL TE with fallbacks (transformers version differences)
try:
    from transformers import Qwen3VLForConditionalGeneration
except ImportError:
    try:
        from transformers.models.qwen3_vl import Qwen3VLForConditionalGeneration
    except ImportError:
        from transformers import AutoModelForImageTextToText as Qwen3VLForConditionalGeneration

print("Loading Heretic text encoder:", HERETIC_TE)
text_encoder = Qwen3VLForConditionalGeneration.from_pretrained(
    HERETIC_TE,
    torch_dtype=DTYPE,
)

print("Loading pipeline:", BASE_MODEL)
pipe = QwenImage21Pipeline.from_pretrained(
    BASE_MODEL,
    text_encoder=text_encoder,
    torch_dtype=DTYPE,
)

# Required for free Colab T4 (~15 GB). On large GPUs you could .to("cuda") instead.
pipe.enable_model_cpu_offload()
pipe.set_progress_bar_config(disable=None)

print("Pipeline ready.")


## 4. Gradio UI + public share link

After this cell finishes, look for a line like `Running on public URL: https://xxxx.gradio.live` — that is your temporary public link.


In [ ]:
import random
from PIL import Image
import gradio as gr

# Colab-friendly sizes (NOT official 2048 defaults — those OOMs on free T4)
ASPECT_RATIOS = {
    "1:1 (1024x1024)": (1024, 1024),
    "1:1 (768x768)": (768, 768),
    "16:9 (1280x720)": (1280, 720),
    "9:16 (720x1280)": (720, 1280),
    "4:3 (1152x896)": (1152, 896),
    "3:4 (896x1152)": (896, 1152),
    "3:2 (1152x768)": (1152, 768),
    "2:3 (768x1152)": (768, 1152),
}

TRANSPARENCY_PREFIX = "This is an RGBA image with transparency. "
TRANSPARENCY_SUFFIX = " The image has alpha channel and the background is transparent."

MAX_SEED = 2**31 - 1


def apply_transparency_template(prompt: str, transparent: bool) -> str:
    prompt = (prompt or "").strip()
    if not transparent:
        return prompt
    # Avoid double-wrapping if the user already used the official template
    low = prompt.lower()
    if "rgba image with transparency" in low and "background is transparent" in low:
        return prompt
    return f"{TRANSPARENCY_PREFIX}{prompt}.{TRANSPARENCY_SUFFIX}"


def generate(
    prompt,
    reference_image,
    aspect_ratio,
    steps,
    seed,
    randomize_seed,
    transparent_rgba,
    negative_prompt,
    progress=gr.Progress(track_tqdm=True),
):
    if not prompt or not str(prompt).strip():
        raise gr.Error("Please enter a prompt.")

    if randomize_seed or seed is None:
        seed = random.randint(0, MAX_SEED)
    seed = int(seed)

    final_prompt = apply_transparency_template(str(prompt), bool(transparent_rgba))
    width, height = ASPECT_RATIOS.get(aspect_ratio, (1024, 1024))

    # Generator device: cpu_offload moves modules; seed on CPU is safest across offload setups
    generator = torch.Generator(device="cpu").manual_seed(seed)

    kwargs = dict(
        prompt=final_prompt,
        negative_prompt=negative_prompt or " ",
        width=width,
        height=height,
        num_inference_steps=int(steps),
        generator=generator,
    )

    if reference_image is not None:
        if not isinstance(reference_image, Image.Image):
            reference_image = Image.fromarray(reference_image)
        # Keep alpha if present (editing / RGBA workflows)
        kwargs["image"] = reference_image

    print(f"seed={seed} size={width}x{height} steps={steps}")
    print(f"prompt={final_prompt[:200]}...")

    with torch.inference_mode():
        out = pipe(**kwargs).images[0]

    return out, seed, final_prompt


with gr.Blocks(title="Qwen-Image-2.1 + Heretic TE") as demo:
    gr.Markdown(
        """
        # Qwen-Image-2.1 + Heretic Text Encoder
        Base: `Qwen/Qwen-Image-2.1` | TE: `pottokao/Qwen-Image-2.1-Text-Encoder-Heretic` (transformers bf16, not GGUF)
        """
    )
    with gr.Row():
        with gr.Column(scale=1):
            prompt = gr.Textbox(
                label="Prompt",
                lines=4,
                placeholder='e.g. A neon shop sign that reads "QWEN IMAGE 2.1", rainy night',
            )
            reference_image = gr.Image(
                label="Optional reference image (edit / I2I)",
                type="pil",
                image_mode="RGBA",
            )
            aspect_ratio = gr.Dropdown(
                label="Aspect ratio (Colab-friendly)",
                choices=list(ASPECT_RATIOS.keys()),
                value="1:1 (1024x1024)",
            )
            with gr.Row():
                steps = gr.Slider(1, 50, value=25, step=1, label="Steps")
                seed = gr.Number(value=42, precision=0, label="Seed")
            randomize_seed = gr.Checkbox(label="Randomize seed", value=False)
            transparent_rgba = gr.Checkbox(
                label="Transparent RGBA (prepend official transparency prompt template)",
                value=False,
            )
            negative_prompt = gr.Textbox(label="Negative prompt", value=" ", lines=1)
            run_btn = gr.Button("Generate", variant="primary")
        with gr.Column(scale=1):
            output_image = gr.Image(label="Output", type="pil", format="png")
            used_seed = gr.Number(label="Seed used", precision=0)
            used_prompt = gr.Textbox(label="Final prompt sent to model", lines=3)

    run_btn.click(
        fn=generate,
        inputs=[
            prompt,
            reference_image,
            aspect_ratio,
            steps,
            seed,
            randomize_seed,
            transparent_rgba,
            negative_prompt,
        ],
        outputs=[output_image, used_seed, used_prompt],
    )
    prompt.submit(
        fn=generate,
        inputs=[
            prompt,
            reference_image,
            aspect_ratio,
            steps,
            seed,
            randomize_seed,
            transparent_rgba,
            negative_prompt,
        ],
        outputs=[output_image, used_seed, used_prompt],
    )

# share=True -> temporary public URL (printed below). Local URL is Colab-internal.
demo.queue(max_size=4).launch(share=True, debug=True)
